In [1]:
!pip -q install requests beautifulsoup4 trafilatura lxml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.7/274.7 kB 21.8 MB/s eta 0:00:00


In [2]:
import os, json, re, datetime, hashlib
from urllib.parse import urlparse

os.makedirs("raw_data", exist_ok=True)
os.makedirs("processed_data", exist_ok=True)
print("Folders ready:", os.listdir("."))


Folders ready: ['.config', 'processed_data', 'raw_data', 'test_set_day_3.txt', 'sample_data']


In [3]:
import requests
from bs4 import BeautifulSoup
import trafilatura

URLS = [
    "https://en.wikipedia.org/wiki/Pittsburgh",
    "https://en.wikipedia.org/wiki/History_of_Pittsburgh",
    "https://www.pittsburghpa.gov/Home",
    "https://www.britannica.com/place/Pittsburgh",
    "https://www.visitpittsburgh.com/",
    "https://www.pittsburghpa.gov/City-Government/Finance-Budget/Taxes/Tax-Forms",
    "https://www.cmu.edu/about/",
    "https://pittsburgh.events/",
    "https://downtownpittsburgh.com/events/",
    "https://community.pghcitypaper.com/pittsburgh/EventSearch?v=d",
    "https://events.cmu.edu/",
    "https://www.cmu.edu/engage/events",
    "https://www.pittsburghsymphony.org/",
    "https://pittsburghopera.org/",
    "https://trustarts.org/",
    "https://carnegiemuseums.org/",
    "https://www.heinzhistorycenter.org/",
    "https://www.thefrickpittsburgh.org/",
    "https://en.wikipedia.org/wiki/List_of_museums_in_Pittsburgh",
    "https://www.visitpittsburgh.com/events-festivals/food-festivals/",
    "https://www.picklesburgh.com/",
    "https://www.pghtacofest.com/",
    "https://pittsburghrestaurantweek.com/",
    "https://littleitalydays.com/",
    "https://bananasplitfest.com/",
    "https://www.visitpittsburgh.com/things-to-do/pittsburgh-sports-teams/",
    "https://www.mlb.com/pirates",
    "https://www.steelers.com/",
    "https://www.nhl.com/penguins/",
    "https://en.wikipedia.org/wiki/International_Conference_on_Machine_Learning",
    "https://en.wikipedia.org/wiki/PPG_Paints_Arena",
    "https://en.wikipedia.org/wiki/Picklesburgh",
    "https://www.cmu.edu/about/history.html",
    "https://en.wikipedia.org/wiki/Carnegie_Mellon_University",
    "https://www.cmu.edu/student-affairs/",
    "https://www.cmu.edu/student-affairs/traditions/",
    "https://www.cmu.edu/student-affairs/scotty/",
    "https://thetartan.org/",
    "https://www.ri.cmu.edu/",
    "https://en.wikipedia.org/wiki/Navlab",
    "https://en.wikipedia.org/wiki/DARPA_Grand_Challenge",
    "https://en.wikipedia.org/wiki/Libratus",
    "https://www.cs.cmu.edu/",
    "https://www.lti.cs.cmu.edu/",
    "https://en.wikipedia.org/wiki/Speech_recognition",
    "https://en.wikipedia.org/wiki/Uber_ATG",
    "https://aptekapgh.com/",
    "https://www.millieshomemade.com/",
    "https://www.klavonsicecream.com/",
    "https://www.wholey.com/",
    "https://en.wikipedia.org/wiki/United_States_Steel",
    "https://en.wikipedia.org/wiki/Alcoa",
    "https://en.wikipedia.org/wiki/Economy_of_Pittsburgh",
    "https://en.wikipedia.org/wiki/History_of_Pittsburgh",
    "https://en.wikipedia.org/wiki/Pittsburgh_Steelers",
    "https://downtownpittsburgh.com/market-square/",
    "https://en.wikipedia.org/wiki/Carnegie_Mellon_University#Campus",
    "https://www.cs.cmu.edu/mobot",
    "https://www.cs.cmu.edu/afs/cs/user/msiegler/www/ASR/futureofcmu-final.html",
    "https://www.ri.cmu.edu/autonomous-vehicles",
    "https://www.cmu.edu/news/stories/archives/2021/august/robotwits-helps-pave-waymos-way.html",
    "https://labs.ri.cmu.edu/av-center",
    "https://www.cmu.edu/news/stories/archives/2017/december/ai-inner-workings.html",
    "https://en.wikipedia.org/wiki/Wholey%27s",
    "https://downtownpittsburgh.com/neighborhoods/market-square/",
    "https://downtownpittsburgh.com/neighborhoods/market-square/",
    "https://www.visitpittsburgh.com/blog/pittsburgh-record-stores/",
    "https://www.visitpittsburgh.com/things-to-do/pittsburgh-sports-teams/",
    "https://www.britannica.com/event/Whiskey-Rebellion",
    "https://www.britannica.com/place/Pittsburgh",
    "https://en.wikipedia.org/wiki/Super_Bowl_XIV",
    "https://en.wikipedia.org/wiki/Roberto_Clemente_Bridge",


]

def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_title_from_html(html: str):
    soup = BeautifulSoup(html, "html.parser")
    if soup.title and soup.title.text:
        return soup.title.text.strip()
    return ""

def domain_of(url: str):
    return urlparse(url).netloc.replace("www.", "")

def make_doc_id(url: str):
    h = hashlib.md5(url.encode("utf-8")).hexdigest()[:10]
    return f"doc_{h}"

records = []
failed = []

for url in URLS:
    try:
        r = requests.get(url, timeout=20, headers={"User-Agent":"Mozilla/5.0"})
        r.raise_for_status()
        html = r.text

        extracted = trafilatura.extract(html, include_comments=False, include_tables=False)
        text = extracted if extracted else ""

        if len(text) < 200:
            soup = BeautifulSoup(html, "html.parser")
            text = soup.get_text(" ", strip=True)

        text = clean_text(text)
        title = get_title_from_html(html)

        if len(text) < 200:
            failed.append((url, "too short after extraction"))
            continue

        rec = {
            "doc_id": make_doc_id(url),
            "url": url,
            "title": title,
            "source": domain_of(url),
            "crawl_time": datetime.date.today().isoformat(),
            "text": text
        }
        records.append(rec)

    except Exception as e:
        failed.append((url, str(e)))

# 保存为 jsonl
out_path = "processed_data/documents.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Saved {len(records)} docs to {out_path}")
print(f"Failed {len(failed)} urls")
for x in failed[:10]:
    print(" -", x)


Saved 59 docs to processed_data/documents.jsonl
Failed 13 urls
 - ('https://www.pittsburghpa.gov/Home', '403 Client Error: Forbidden for url: https://www.pittsburghpa.gov/Home')
 - ('https://www.britannica.com/place/Pittsburgh', '403 Client Error: Forbidden for url: https://www.britannica.com/place/Pittsburgh')
 - ('https://www.pittsburghpa.gov/City-Government/Finance-Budget/Taxes/Tax-Forms', '403 Client Error: Forbidden for url: https://www.pittsburghpa.gov/City-Government/Finance-Budget/Taxes/Tax-Forms')
 - ('https://pittsburgh.events/', "('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))")
 - ('https://community.pghcitypaper.com/pittsburgh/EventSearch?v=d', '403 Client Error: Forbidden for url: https://community.pghcitypaper.com/pittsburgh/EventSearch?v=d')
 - ('https://www.pittsburghsymphony.org/', '403 Client Error: Forbidden for url: https://www.pittsburghsymphony.org/')
 - ('https://trustarts.org/', '403 Client Error: Forbidden for url: 

In [4]:
!pip -q install rank-bm25 nltk

In [5]:
import os

print("Current working dir:", os.getcwd())
print("\nTop-level files/folders:")
print(os.listdir("."))

if os.path.exists("processed_data"):
    print("\nprocessed_data contents:")
    print(os.listdir("processed_data"))
else:
    print("\nprocessed_data folder does NOT exist.")


Current working dir: /content

Top-level files/folders:
['.config', 'processed_data', 'raw_data', 'test_set_day_3.txt', 'sample_data']

processed_data contents:
['documents.jsonl']


In [6]:
import json, os, re
from pathlib import Path

os.makedirs("chunks", exist_ok=True)

def split_into_word_chunks(text, chunk_size=220, overlap=40):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk_words = words[start:end]
        chunk_text = " ".join(chunk_words).strip()
        if len(chunk_text) > 80:
            chunks.append((start, end, chunk_text))
        if end == len(words):
            break
        start = end - overlap
    return chunks

in_path = "processed_data/documents.jsonl"
out_path = "chunks/chunks.jsonl"

chunk_count = 0
doc_count = 0

with open(in_path, "r", encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
    for line in fin:
        doc = json.loads(line)
        doc_count += 1
        doc_id = doc["doc_id"]
        text = re.sub(r"\s+", " ", doc.get("text", "")).strip()

        doc_chunks = split_into_word_chunks(text, chunk_size=220, overlap=40)
        for i, (s, e, ctext) in enumerate(doc_chunks):
            rec = {
                "chunk_id": f"{doc_id}_chunk_{i:04d}",
                "doc_id": doc_id,
                "title": doc.get("title", ""),
                "url": doc.get("url", ""),
                "source": doc.get("source", ""),
                "word_start": s,
                "word_end": e,
                "text": ctext
            }
            fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
            chunk_count += 1

print(f"Processed docs: {doc_count}")
print(f"Saved chunks: {chunk_count}")
print(f"Output: {out_path}")


Processed docs: 59
Saved chunks: 896
Output: chunks/chunks.jsonl


In [7]:
import json, re
from rank_bm25 import BM25Okapi

# 读取 chunks
chunks = []
with open("chunks/chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print("Loaded chunks:", len(chunks))

import re

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    return text.split()

def expand_query(query: str) -> str:
    q = query.lower()
    extra = []
    if "carnegie mellon" in q or "cmu" in q:
        extra += ["carnegie mellon university", "founded", "established", "history", "university"]
    if "icml" in q or ("machine learning" in q and "1980" in q):
        extra += ["ICML", "International Conference on Machine Learning", "first conference", "1980", "Pittsburgh"]
    return query + " " + " ".join(extra)

def rerank_for_query(query, candidates):
    q = query.lower()
    q_tokens = set(tokenize(query))

    reranked = []
    for c in candidates:
        url = (c.get("url", "") or "").lower()
        text = ((c.get("title","") or "") + " " + (c.get("text","") or "")).lower()
        t_tokens = set(tokenize(text))

        overlap = len(q_tokens & t_tokens)
        score = overlap * 3

        if ("cmu" in q or "carnegie mellon" in q):
            if "cmu.edu" in url: score += 8
            if "wikipedia.org/wiki/carnegie_mellon_university" in url: score += 6
            if "events.cmu.edu" in url: score -= 3

        if "pittsburgh" in q:
            if "wikipedia.org/wiki/pittsburgh" in url: score += 8

        if any(w in q for w in ["event", "festival", "concert", "perform", "schedule", "upcoming", "arena", "venue"]):
            if any(w in text for w in ["event", "festival", "concert", "perform", "schedule", "upcoming", "arena", "venue"]):
                score += 4

        reranked.append((score, c))

    reranked.sort(key=lambda x: x[0], reverse=True)
    return [c for _, c in reranked]

tokenized_corpus = [tokenize(c["text"]) for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

def sparse_retrieve(query, topk=6, recall_k=30):
    q2 = expand_query(query)
    q_tokens = tokenize(q2)
    scores = bm25.get_scores(q_tokens)
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:recall_k]
    candidates = [chunks[i] for i in top_idx]
    reranked = rerank_for_query(query, candidates)
    return reranked[:topk]

def bm25_retrieve(query, topk=50):
    q2 = expand_query(query)
    q_tokens = tokenize(q2)
    scores = bm25.get_scores(q_tokens)
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:topk]
    results = []
    for i in top_idx:
        rec = dict(chunks[i])
        rec["bm25_score"] = float(scores[i])
        results.append(rec)
    return results


Loaded chunks: 896


In [8]:
def rrf_fuse(bm25_list, dense_list, topk=6, k=60, w_bm25=1.0, w_dense=1.0):
    """
    Reciprocal Rank Fusion:
    score(d) = Σ w / (k + rank)
    ranks are 1-based
    """
    scores = {}

    # bm25 contribution
    for rank, item in enumerate(bm25_list, start=1):
        cid = item["chunk_id"]
        scores.setdefault(cid, {"rrf": 0.0, "item": item})
        scores[cid]["rrf"] += w_bm25 / (k + rank)

    # dense contribution
    for rank, item in enumerate(dense_list, start=1):
        cid = item["chunk_id"]
        scores.setdefault(cid, {"rrf": 0.0, "item": item})
        scores[cid]["rrf"] += w_dense / (k + rank)

    fused = []
    for cid, obj in scores.items():
        rec = dict(obj["item"])
        rec["rrf_score"] = float(obj["rrf"])
        fused.append(rec)

    fused.sort(key=lambda x: x["rrf_score"], reverse=True)
    return fused[:topk]

In [9]:
!pip -q install -U sentence-transformers faiss-cpu

import os, json, numpy as np
import faiss
from sentence_transformers import SentenceTransformer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.2 MB/s eta 0:00:00


In [10]:
chunks = []
with open("chunks/chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print("Loaded chunks:", len(chunks))
print("Example chunk keys:", list(chunks[0].keys()))

Loaded chunks: 896
Example chunk keys: ['chunk_id', 'doc_id', 'title', 'url', 'source', 'word_start', 'word_end', 'text']


In [11]:
# settings
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 64
INDEX_DIR = "dense_index"
os.makedirs(INDEX_DIR, exist_ok=True)

emb_path = os.path.join(INDEX_DIR, "chunk_emb.npy")
meta_path = os.path.join(INDEX_DIR, "chunk_meta.jsonl")
faiss_path = os.path.join(INDEX_DIR, "faiss.index")

# oad embedding model
encoder = SentenceTransformer(EMB_MODEL)
print("Encoder:", EMB_MODEL)

# build embeddings
texts = [c["text"] for c in chunks]

emb = encoder.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True
)

emb = np.asarray(emb, dtype=np.float32)
print("Embeddings shape:", emb.shape, emb.dtype)

#build FAISS index (cosine via inner product + normalized vectors)
dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)  # inner product
index.add(emb)

print("FAISS ntotal:", index.ntotal)

np.save(emb_path, emb)

with open(meta_path, "w", encoding="utf-8") as f:
    for c in chunks:
        keep = {
            "chunk_id": c.get("chunk_id"),
            "doc_id": c.get("doc_id"),
            "title": c.get("title"),
            "url": c.get("url"),
            "source": c.get("source"),
            "text": c.get("text")
        }
        f.write(json.dumps(keep, ensure_ascii=False) + "\n")

faiss.write_index(index, faiss_path)

print("Saved:")
print("-", emb_path)
print("-", meta_path)
print("-", faiss_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoder: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Embeddings shape: (896, 384) float32
FAISS ntotal: 896
Saved:
- dense_index/chunk_emb.npy
- dense_index/chunk_meta.jsonl
- dense_index/faiss.index


In [12]:
import numpy as np
import faiss
import json
import os

INDEX_DIR = "dense_index"
meta_path = os.path.join(INDEX_DIR, "chunk_meta.jsonl")
faiss_path = os.path.join(INDEX_DIR, "faiss.index")

# load meta
dense_chunks = []
with open(meta_path, "r", encoding="utf-8") as f:
    for line in f:
        dense_chunks.append(json.loads(line))

# load index
dense_index = faiss.read_index(faiss_path)

print("Dense chunks:", len(dense_chunks))
print("FAISS ntotal:", dense_index.ntotal)

try:
    encoder
except NameError:
    from sentence_transformers import SentenceTransformer
    EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
    encoder = SentenceTransformer(EMB_MODEL)

def dense_retrieve(query, topk=6):
    q_emb = encoder.encode([query], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype=np.float32)

    scores, idx = dense_index.search(q_emb, topk)
    idx = idx[0].tolist()
    scores = scores[0].tolist()

    results = []
    for i, s in zip(idx, scores):
        if i < 0:
            continue
        rec = dict(dense_chunks[i])
        rec["dense_score"] = float(s)
        results.append(rec)
    return results

# quick test
q = "Who is Pittsburgh named after?"
res = dense_retrieve(q, topk=5)
print("Top dense:", res[0]["url"], res[0]["dense_score"])

Dense chunks: 896
FAISS ntotal: 896
Top dense: https://en.wikipedia.org/wiki/Pittsburgh 0.7309312224388123


In [13]:
def hybrid_retrieve(query, topk=6, bm25_k=50, dense_k=50, rrf_k=60,
                    w_bm25=1.0, w_dense=1.0, use_heuristic_rerank=True):
    # 1) get candidate lists
    bm25_list = bm25_retrieve(query, topk=bm25_k)
    dense_list = dense_retrieve(query, topk=dense_k)

    # 2) fuse
    fused = rrf_fuse(
        bm25_list=bm25_list,
        dense_list=dense_list,
        topk=max(topk, 20),   # 先多拿点给 rerank
        k=rrf_k,
        w_bm25=w_bm25,
        w_dense=w_dense
    )

    # 3) optional heuristic rerank (generic)
    if use_heuristic_rerank:
        fused = rerank_for_query(query, fused)

    return fused[:topk]

In [14]:
test_q = "What famous machine learning venue had its first conference in Pittsburgh in 1980?"

print("\n[BM25 top]")
bm = hybrid_retrieve(test_q, topk=3)
for i, r in enumerate(bm, 1):
    print(i, r["url"])

print("\n[Dense top]")
dn = dense_retrieve(test_q, topk=3)
for i, r in enumerate(dn, 1):
    print(i, r["url"], r["dense_score"])


[BM25 top]
1 https://en.wikipedia.org/wiki/International_Conference_on_Machine_Learning
2 https://en.wikipedia.org/wiki/Pittsburgh
3 https://en.wikipedia.org/wiki/History_of_Pittsburgh

[Dense top]
1 https://en.wikipedia.org/wiki/Carnegie_Mellon_University#Campus 0.5316902995109558
2 https://en.wikipedia.org/wiki/Carnegie_Mellon_University 0.5316902995109558
3 https://en.wikipedia.org/wiki/Pittsburgh 0.5148754119873047


In [15]:
import re

def answer_from_retrieved(query, retrieved):
    """
    Generic, non-overfitting answer extractor.
    - No hard-coded QA pairs
    - Light heuristics for a few common question types
    - Otherwise return a short span from top retrieved chunk
    """
    q = query.strip().lower()
    if not retrieved:
        return "Unknown"

    # Build a compact context from top chunks
    context = " ".join([r["text"] for r in retrieved[:3]])
    context_clean = re.sub(r"\s+", " ", context).strip()

    # 1) "When/What year" questions -> extract a 4-digit year near keywords
    if any(w in q for w in ["when", "what year", "founded", "established", "opened", "built", "created", "born"]):
        # look for "founded in 1900" / "established in 18xx"
        m = re.search(r"\b(?:founded|established|opened|built|created)\b[^.]{0,60}\b(1[6-9]\d{2}|20\d{2})\b",
                      context_clean, flags=re.IGNORECASE)
        if m:
            return m.group(1)
        # fallback: any year in context
        m2 = re.search(r"\b(1[6-9]\d{2}|20\d{2})\b", context_clean)
        if m2:
            return m2.group(1)

    # 2) "Who" questions -> try to extract a capitalized name phrase after cues
    if q.startswith("who") or "who " in q:
        # common patterns: "named after X", "in honor of X"
        m = re.search(r"\b(?:named\s+(?:after|in honor of)|in honor of)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,3})\b",
                      context, flags=re.IGNORECASE)
        if m:
            return m.group(1).strip()

    # 3) "What is the name of" -> try quoted title or Capitalized phrase
    if "what is the name of" in q or q.startswith("what is"):
        # quoted phrase first
        mq = re.search(r"“([^”]{3,80})”|\"([^\"]{3,80})\"", context)
        if mq:
            return (mq.group(1) or mq.group(2)).strip()

    # 4) Default: return the first good sentence from top chunk (short)
    top_text = retrieved[0]["text"]
    sents = re.split(r'(?<=[.!?])\s+', top_text.strip())
    for s in sents:
        s = s.strip()
        if 20 <= len(s) <= 180:
            return s

    # fallback: truncate
    return top_text[:160].strip()

def answer_question(query, topk=6):
    retrieved = sparse_retrieve(query, topk=topk)
    ans = answer_from_retrieved(query, retrieved)
    return ans, retrieved

In [16]:
!pip -q install transformers accelerate sentencepiece

In [17]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

READER_MODEL = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(READER_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(READER_MODEL)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print("Reader device:", device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Reader device: cuda


In [18]:
def build_prompt(question, retrieved, max_chars=3500):
    parts = []
    for i, c in enumerate(retrieved, 1):
        parts.append(f"[Doc {i}] {c['text'].strip()}")
    context = "\n".join(parts)[:max_chars]

    prompt = (
        "Answer the question using ONLY the documents.\n"
        "If the answer is not explicitly stated in the documents, output exactly: Unknown\n"
        "Output ONLY the answer, no extra words.\n\n"
        f"Question: {question}\n\n"
        f"Documents:\n{context}\n\n"
        "Answer:"
    )
    return prompt

In [19]:
def normalize_unknown(ans: str) -> str:
    a = ans.strip().lower()
    if a in {"unknown", "unanswerable", "not answerable", "cannot be determined", "not enough information"}:
        return "Unknown"
    return ans.strip()

In [20]:
def rag_answer(question, topk=6, max_new_tokens=32):
    retrieved = hybrid_retrieve(question, topk=topk)
    prompt = build_prompt(question, retrieved)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
    ans = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    ans = ans.split("\n")[0].strip()
    ans = normalize_unknown(ans)
    return ans, retrieved

In [21]:
questions = {
  "1": "Who is Pittsburgh named after?",
  "2": "When was Carnegie Mellon University founded?",
  "3": "What famous machine learning venue had its first conference in Pittsburgh in 1980?"
}
print("Loaded questions:", len(questions))


Loaded questions: 3


In [22]:
# Run RAG on TXT test set
import os, json

TEST_TXT_PATH = "/content/test_set_day_3.txt"
OUT_JSON_PATH = "outputs/system_output_1.json"

os.makedirs("outputs", exist_ok=True)

# --- load questions (one question per line) ---
questions = []
with open(TEST_TXT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        q = line.strip()
        if q:
            questions.append(q)

print("Loaded test questions:", len(questions))
print("Example:", questions[0] if questions else "EMPTY")

# --- run inference ---
preds = {}
for i, q in enumerate(questions, start=1):
    ans, _ = rag_answer(q, topk=6)   # use your existing rag_answer()
    preds[str(i)] = ans
    if i <= 5:
        print(f"[{i}] Q:", q)
        print("   A:", ans)

# --- save outputs ---
with open(OUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False, indent=2)

print("Saved:")
print("-", OUT_JSON_PATH)

Loaded test questions: 400
Example: How many Division I varsity teams does the University of Pittsburgh sponsor?
[1] Q: How many Division I varsity teams does the University of Pittsburgh sponsor?
   A: Three
[2] Q: What are Pittsburgh's team colors?
   A: black and gold
[3] Q: What are the traditional colors for Pittsburgh sports teams?
   A: Black and gold
[4] Q: What rivers form the Ohio River in Pittsburgh?
   A: Allegheny and Monongahela Rivers
[5] Q: Where is the Orwell film playing in October?
   A: Pittsburgh
Saved:
- outputs/system_output_1.json
